# ML-06 — Signal Audit: Do the Flags Hold?

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/omi290/flyrank-ml-internship/blob/main/work/notebooks/w04_signal_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Distributions

*Look before deciding: distributions of your key fields. Note the heavy tails.*

# Distribution Analysis

The key search and engagement signals are positively skewed. Most pages receive relatively low traffic and engagement, while a small number of pages account for very high values. This heavy-tailed distribution suggests that bucketing or normalization is preferable to using raw values directly in rule-based systems.

In [1]:
!pip -q install duckdb huggingface_hub pyarrow

In [2]:
from google.colab import userdata
from huggingface_hub import hf_hub_download
import duckdb

HF_TOKEN = userdata.get("HF_TOKEN")

In [3]:
march_path = hf_hub_download(
    repo_id="FlyRank/internship-warehouse",
    repo_type="dataset",
    filename="fact_content_daily_performance/month=2026-03/data_0.parquet",
    token=HF_TOKEN,
)

print(march_path)

fact_content_daily_performance/month=202(…): reconstructing file:   0%|          |  0.00B /  124MB            

fact_content_daily_performance/month=202(…): downloading bytes:           |  0.00B            

/root/.cache/huggingface/hub/datasets--FlyRank--internship-warehouse/snapshots/50cbf7c3909d07be4d1b5906b4d09e882e5acbf2/fact_content_daily_performance/month=2026-03/data_0.parquet


In [4]:
db = duckdb.connect()

db.sql(f"""
CREATE OR REPLACE VIEW march_data AS
SELECT *
FROM read_parquet('{march_path}')
""")

In [5]:
db.sql("""
SELECT *
FROM march_data
LIMIT 5
""").df()

,report_date,client_hash_id,content_hash_id,client_has_gsc,client_has_ga4,gsc_data_available,ga4_data_available,gsc_impressions,gsc_clicks,gsc_sum_position,...,sessions_ai,ai_chatgpt,ai_perplexity,ai_gemini,ai_copilot,ai_claude,ai_meta,ai_other,scroll_events,month
0,2026-03-01,client_73cda7b4e4f265ea,content_b7e512995f79d5a6,True,False,True,<NA>,20,0,67,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,2026-03
1,2026-03-01,client_73cda7b4e4f265ea,content_05597932fe4da067,True,False,True,<NA>,1,0,0,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,2026-03
2,2026-03-01,client_73cda7b4e4f265ea,content_7a105f548d9c6916,True,False,True,<NA>,125,1,616,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,2026-03
3,2026-03-01,client_73cda7b4e4f265ea,content_905aa32a0230694e,True,False,True,<NA>,7,0,28,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,2026-03
4,2026-03-01,client_73cda7b4e4f265ea,content_a3ea9792f793ec72,True,False,True,<NA>,11,0,25,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,2026-03


In [6]:
db.sql("""
SELECT
MIN(gsc_impressions) AS min_impressions,
AVG(gsc_impressions) AS avg_impressions,
MAX(gsc_impressions) AS max_impressions,

MIN(gsc_clicks) AS min_clicks,
AVG(gsc_clicks) AS avg_clicks,
MAX(gsc_clicks) AS max_clicks,

MIN(ga4_pageviews) AS min_pageviews,
AVG(ga4_pageviews) AS avg_pageviews,
MAX(ga4_pageviews) AS max_pageviews

FROM march_data
""").df()

,min_impressions,avg_impressions,max_impressions,min_clicks,avg_clicks,max_clicks,min_pageviews,avg_pageviews,max_pageviews
0,0,28.518119,40084,0,0.083508,274,0,0.217637,875


## 2. Signal test #1 / #2 / #3 (verdict each)

*Three safe signals, each with a mini-test and a verdict: CONFIRMED / OPPOSITE / MIXED / FALSE.*

# Signal Tests

Three relationships were tested using historical March 2026 data.

The goal is to determine whether these signals are reliable enough to support a rule-based baseline.

In [7]:
db.sql("""
SELECT
CASE
WHEN gsc_avg_position<=10 THEN 'Top 10'
WHEN gsc_avg_position<=20 THEN '11-20'
WHEN gsc_avg_position<=30 THEN '21-30'
ELSE '30+'
END position_bucket,

COUNT(*) AS n,

AVG(ga4_pageviews) AS avg_pageviews

FROM march_data

WHERE ga4_data_available IS TRUE

GROUP BY 1

ORDER BY 1
""").df()

,position_bucket,n,avg_pageviews
0,11-20,75610,4.211506
1,21-30,53699,4.878936
2,30+,96182,3.166746
3,Top 10,188475,3.182629


In [8]:
db.sql("""
SELECT

CASE

WHEN gsc_impressions<100 THEN 'Low'
WHEN gsc_impressions<1000 THEN 'Medium'
ELSE 'High'

END impression_bucket,

COUNT(*) AS n,

AVG(gsc_clicks) AS avg_clicks

FROM march_data

GROUP BY 1

ORDER BY 1
""").df()

,impression_bucket,n,avg_clicks
0,High,32419,5.068787
1,Low,9202770,0.018722
2,Medium,606189,0.800425


### Signal Test 3

**Signal:** GA4 Sessions vs Scroll Events

**Verdict:** **CONFIRMED**

The bucket analysis shows a clear positive relationship between GA4 sessions and scroll events. Pages with higher session counts consistently generate more scroll activity than pages with medium or low session counts. This supports using historical engagement metrics as reliable inputs for rule-based prioritization and future machine learning models.

In [9]:
db.sql("""
SELECT

CASE

WHEN ga4_sessions<10 THEN 'Low'
WHEN ga4_sessions<100 THEN 'Medium'
ELSE 'High'

END session_bucket,

COUNT(*) AS n,

AVG(scroll_events) AS avg_scroll

FROM march_data

WHERE ga4_data_available IS TRUE

GROUP BY 1

ORDER BY 1
""").df()

,session_bucket,n,avg_scroll
0,High,299,23.314381
1,Low,389458,0.346333
2,Medium,24209,3.232269


## 3. The flag-linked test

*Pick a signal one of FlyRank's real flags relies on. Does the data support the rule's assumption?*

# Flag-linked Test

The relationship between search impressions and clicks was selected because it is directly related to FlyRank's visibility-based prioritization logic.

The observed data confirmed that higher search visibility is associated with higher click volume, supporting the assumption behind visibility-driven content recommendations.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 4. What this means in practice

*Two or three sentences: what a content team should take from this.*

## Practical Takeaways

The analysis shows that historical engagement and search visibility signals are reliable indicators for prioritizing content refresh opportunities. Impressions, clicks, sessions, and scroll events exhibit consistent relationships, while average search position alone is less reliable. A rule-based system should combine multiple historical signals rather than relying on a single metric when deciding which content pages to prioritize.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.